# 08 — Global Explanations

Global XAI methods describe average model behaviour across the input space, rather than individual predictions.

## PDP, ALE, and Global Surrogates

**Partial Dependence Plots (PDP)**: show the marginal effect of one or two features on the prediction, averaging over the marginal distribution of all other features. The PDP for feature *x_j* is:
$$\hat{f}_{x_j}(x_j) = \mathbb{E}_{X_{-j}}[f(x_j, X_{-j})]$$
Problem: if features are correlated, this marginalises over combinations of features that don't exist in the training data (extrapolation artefacts).

**Accumulated Local Effects (ALE)**: fix the correlation problem by instead computing *differences* in predictions within small intervals, then accumulating:
$$\hat{f}^{ALE}_j(x) = \int_{z_{0,j}}^x \mathbb{E}[\partial f / \partial x_j | x_j = z] dz$$
ALE averages over the conditional distribution (not the marginal), so it only uses data points that actually exist.

**Global Surrogates**: train a simple, interpretable model (decision tree, linear model) on the original model's predictions across the full training set. The surrogate is globally interpretable but may not faithfully represent the model in any local region.

In [ ]:
# PDP + ALE comparison
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.datasets import make_regression

np.random.seed(42)

# Correlated features make PDP vs ALE differences visible
n = 1000
X1 = np.random.randn(n)
X2 = X1 * 0.8 + np.random.randn(n) * 0.3  # correlated with X1
X3 = np.random.randn(n)
X = np.column_stack([X1, X2, X3])
y = 2 * X1 + X2**2 + np.random.randn(n) * 0.5

gbm = GradientBoostingRegressor(n_estimators=100, random_state=42)
gbm.fit(X, y)
print(f'GBM R2: {gbm.score(X, y):.3f}')

def compute_pdp(model, X, feature_idx, n_grid=50):
    grid = np.linspace(X[:, feature_idx].min(), X[:, feature_idx].max(), n_grid)
    pdp_vals = []
    for val in grid:
        X_copy = X.copy()
        X_copy[:, feature_idx] = val
        pdp_vals.append(model.predict(X_copy).mean())
    return grid, np.array(pdp_vals)

def compute_ale(model, X, feature_idx, n_bins=20):
    feat = X[:, feature_idx]
    percentiles = np.percentile(feat, np.linspace(0, 100, n_bins + 1))
    ale_vals = []
    bin_centers = []
    for k in range(n_bins):
        lo, hi = percentiles[k], percentiles[k+1]
        mask = (feat >= lo) & (feat <= hi)
        if mask.sum() < 2:
            continue
        X_lo = X[mask].copy(); X_lo[:, feature_idx] = lo
        X_hi = X[mask].copy(); X_hi[:, feature_idx] = hi
        local_effect = (model.predict(X_hi) - model.predict(X_lo)).mean()
        ale_vals.append(local_effect)
        bin_centers.append((lo + hi) / 2)

    ale_cumulative = np.cumsum(ale_vals)
    ale_cumulative -= ale_cumulative.mean()  # center
    return np.array(bin_centers), ale_cumulative

pdp_grid, pdp_vals = compute_pdp(gbm, X, feature_idx=1)
ale_grid, ale_vals = compute_ale(gbm, X, feature_idx=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(pdp_grid, pdp_vals - pdp_vals.mean(), color='steelblue')
ax1.set_xlabel('X2'); ax1.set_ylabel('PDP (centered)'); ax1.set_title('Partial Dependence Plot')
ax2.plot(ale_grid, ale_vals, color='tomato')
ax2.set_xlabel('X2'); ax2.set_ylabel('ALE'); ax2.set_title('Accumulated Local Effects')
plt.suptitle('PDP vs ALE — correlated features')
plt.tight_layout()
plt.savefig('/tmp/pdp_ale.png', dpi=100, bbox_inches='tight')
plt.show()
print('PDP vs ALE plot saved')

In [ ]:
# Global surrogate model
from sklearn.tree import DecisionTreeRegressor, export_text

# Train a decision tree to approximate the GBM
surrogate = DecisionTreeRegressor(max_depth=3, random_state=42)
surrogate.fit(X, gbm.predict(X))

# How faithful is the surrogate?
surr_preds = surrogate.predict(X)
gbm_preds = gbm.predict(X)
r2_fidelity = 1 - np.sum((gbm_preds - surr_preds)**2) / np.sum((gbm_preds - gbm_preds.mean())**2)
print(f'Surrogate fidelity R2: {r2_fidelity:.3f}')
print(f'Surrogate vs GBM agreement: {r2_fidelity:.1%} of variance explained')

print('\nDecision tree structure (global surrogate):')
print(export_text(surrogate, feature_names=['X1', 'X2', 'X3']))